# Ch.11 — Fine-tuned MentalBERT Training

**TP2 upgrade**: End-to-end fine-tuning of `mental/mental-bert-base-uncased`  
on `dair-ai/emotion` (16,000 train / 2,000 val / 2,000 test).

**Runtime**: GPU (T4 recommended) · ~20–30 min

**Connects to**:
- Ch.11 (*Hands-On LLMs*): frozen → full fine-tuning of representation models
- Ch.10 (*Hands-On LLMs*): bi-encoder concept → per-sentence encoding + mean pooling

**Output**: saves fine-tuned model to `saved_models/ch11_mentalbert/`

## 0. Setup

In [ ]:
# Clone repo (skip if already cloned)
import os
if not os.path.exists('mental-health-classifier'):
    !git clone https://github.com/CJiu01/mental-health-classifier.git
%cd mental-health-classifier

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn nltk

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('dair-ai/emotion')
EMOTION_LABELS = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
NUM_LABELS     = len(EMOTION_LABELS)

print('Train:', len(dataset['train']))
print('Val  :', len(dataset['validation']))
print('Test :', len(dataset['test']))
print('Sample:', dataset['train'][0])

## 2. Tokenise

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'mental/mental-bert-base-uncased'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print('Tokenisation complete.')

## 3. Model

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {i: l for i, l in enumerate(EMOTION_LABELS)}
label2id = {l: i for i, l in enumerate(EMOTION_LABELS)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Training

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

SAVE_DIR = 'saved_models/ch11_mentalbert'

training_args = TrainingArguments(
    output_dir                  = SAVE_DIR,
    num_train_epochs            = 4,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    fp16                        = True,
    logging_steps               = 50,
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized['train'],
    eval_dataset    = tokenized['validation'],
    compute_metrics = compute_metrics,
)

trainer.train()

## 5. Test Set Evaluation

In [ ]:
from sklearn.metrics import classification_report

# 6-emotion classification
test_results = trainer.predict(tokenized['test'])
preds        = np.argmax(test_results.predictions, axis=-1)
labels       = test_results.label_ids

print('=== Ch.11 — 6-Emotion Classification (Test Set) ===')
print(classification_report(labels, preds, target_names=EMOTION_LABELS))

In [ ]:
import sys
sys.path.insert(0, '.')
from config import RISK_THRESHOLDS
import torch.nn.functional as F
import torch

def compute_risk(emotion_scores):
    risk_score = emotion_scores['sadness'] + emotion_scores['fear']
    joy_score  = emotion_scores['joy']     + emotion_scores['love']
    if   risk_score > RISK_THRESHOLDS['crisis']  : level = 'Crisis'
    elif risk_score > RISK_THRESHOLDS['negative'] : level = 'Negative'
    elif joy_score  > RISK_THRESHOLDS['positive'] : level = 'Positive'
    else                                          : level = 'Neutral'
    return level

RISK_FROM_EMOTION = {
    'sadness': 'Negative', 'fear': 'Negative',
    'joy': 'Positive',     'love': 'Positive',
    'anger': 'Neutral',    'surprise': 'Neutral',
}

# 4-tier risk classification
probs_all   = F.softmax(torch.tensor(test_results.predictions), dim=-1).numpy()
true_risks  = [RISK_FROM_EMOTION[EMOTION_LABELS[l]] for l in labels]
pred_risks  = []
for probs in probs_all:
    scores = {n: float(p) for n, p in zip(EMOTION_LABELS, probs)}
    pred_risks.append(compute_risk(scores))

print('=== Ch.11 — 4-Tier Risk Classification (Test Set) ===')
print(classification_report(
    true_risks, pred_risks,
    labels=['Positive', 'Neutral', 'Negative', 'Crisis']
))

## 6. Save Model

In [ ]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model saved to {SAVE_DIR}/')
print('Files:', os.listdir(SAVE_DIR))

## 7. Download to Local

Run the cell below to zip and download the fine-tuned model.

In [ ]:
import shutil
from google.colab import files

zip_path = 'ch11_mentalbert.zip'
shutil.make_archive('ch11_mentalbert', 'zip', SAVE_DIR)
files.download(zip_path)
print('Download started.')

## 8. Confusion Matrix

Visual result for the TP2 report.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS)
plt.title('Ch.11 Fine-tuned MentalBERT — Confusion Matrix (Test Set)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix_ch11.png', dpi=150)
plt.show()
print('Saved: confusion_matrix_ch11.png')